In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/annotated/checkpoints/pre_cell_6.pickle

trying: ['load_supplier']
me:  6
trying: ['nation']
me:  4
trying: ['pd']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['part']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2
trying: ['load_orders']
me:  2
trying: ['load_nation']
me:  4
trying: ['load_partsupp']
me:  5
trying: ['load_part']
me:  3
trying: ['lineitem']


me:  1
trying: ['partsupp']
me:  5
trying: ['load_lineitem']
me:  1
trying: ['supplier']
me:  6
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable part
Declaring variable load_part
Declaring variable nation
Declaring variable load_nation
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable load_supplier
Declaring variable supplier


In [4]:
%%RecordEvent
%%time
### cell 6 ###
# Filter only the part keys we need
ghost_parts = part.loc[part.P_NAME.str.contains("ghost"), ["P_PARTKEY"]]

# Chain merges while selecting only the necessary columns at each step
tmp = (
    lineitem
    .merge(ghost_parts, left_on="L_PARTKEY", right_on="P_PARTKEY")
    .merge(
        supplier[["S_SUPPKEY", "S_NATIONKEY"]],
        left_on="L_SUPPKEY", right_on="S_SUPPKEY"
    )
    .merge(
        nation[["N_NATIONKEY", "N_NAME"]],
        left_on="S_NATIONKEY", right_on="N_NATIONKEY"
    )
    .merge(
        partsupp[["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]],
        left_on=["L_PARTKEY", "L_SUPPKEY"], right_on=["PS_PARTKEY", "PS_SUPPKEY"]
    )
    .merge(
        orders[["O_ORDERKEY", "O_ORDERDATE"]],
        left_on="L_ORDERKEY", right_on="O_ORDERKEY"
    )
)

# Compute the profit and extract the order year
tmp["TMP"] = (
    tmp.L_EXTENDEDPRICE * (1 - tmp.L_DISCOUNT)
    - tmp.PS_SUPPLYCOST * tmp.L_QUANTITY
)
tmp["O_YEAR"] = tmp.O_ORDERDATE.dt.year

# Group by nation and year, then sum(TMP)
total = tmp.groupby(["N_NAME", "O_YEAR"], as_index=False, sort=False)["TMP"].sum()

# Sort by nation ascending and within each nation by year descending
total = total.sort_values(["N_NAME", "O_YEAR"], ascending=[True, False])

CPU times: user 91.1 ms, sys: 70 ms, total: 161 ms
Wall time: 178 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_6_try_2.pickle

migration speed (bps): 813556854.8101006
---------------------------
variables to migrate:
lineitem 48
tmp 48
ghost_parts 48
tpch_parent 54
load_nation 144
nation 48
total 48
load_partsupp 144
partsupp 48
load_supplier 144
supplier 48
DATA_ROOT 80
part 48
load_part 144
orders 48
load_orders 144
STORAGE_OPTS 64
pd 72
load_lineitem 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_6_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem', 'orders']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['lineitem', 'orders', 'part']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables ['lineitem', 'nation', 'orders', 'part']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['nation', 'orders', 'part', 'partsupp', 'lineitem']
Modified dataframes
======= Cell 6 =======
Input varia

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q09/opt_cell_exec_info_6_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[6], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/annotated/checkpoints/post_cell_6.pickle
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['jn1']
me:  7
trying: ['DATA_ROOT']
me:  0
trying: ['jn4']


me:  7
trying: ['load_part']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['fpart']
me:  7
trying: ['load_nation']
me:  4
trying: ['orig_output']
me:  8
trying: ['nation']
me:  4
trying: ['jn2']
me:  7
trying: ['psel']
me:  7
trying: ['load_partsupp']
me:  5
trying: ['partsupp']


me:  5
trying: ['jn5']


me:  7
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_supplier']
me:  6
trying: ['supplier']
me:  6
trying: ['pd']
me:  0
trying: ['orders']
me:  2
trying: ['load_orders']
me:  2
trying: ['part']
me:  3
trying: ['jn3']


me:  7
trying: ['total']
me:  7
trying: ['gb']
me:  7
trying: ['load_lineitem']
me:  1
trying: ['lineitem']


me:  1


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable load_part
Declaring variable part
Declaring variable load_nation
Declaring variable nation
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable load_supplier
Declaring variable supplier
Declaring variable jn1
Declaring variable jn4
Declaring variable fpart
Declaring variable jn2
Declaring variable psel
Declaring variable jn5
Declaring variable jn3
Declaring variable total
Declaring variable gb
Declaring variable orig_output


ValueError: Content of df1 and df2 do not match